<a href="https://colab.research.google.com/github/andyotani/Machine-learning_26-27/blob/main/JS09/Tugas__Lab_02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Langkah 0: Install Libraries Tambahan
Kita perlu memastikan pustaka `annoy` dan `hnswlib` sudah terinstall di environment Colab.

In [20]:
!pip install annoy hnswlib faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 47.4 MB/s  0:00:00


In [21]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("bwandowando/spotify-songs-with-attributes-and-lyrics")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'spotify-songs-with-attributes-and-lyrics' dataset.
Path to dataset files: /kaggle/input/spotify-songs-with-attributes-and-lyrics


### Langkah 1: Load Data & Preprocessing
Kita akan memuat dataset, menyeleksi fitur numerik yang diminta, dan melakukan standardisasi dengan `StandardScaler`.

In [22]:
import pandas as pd
import numpy as np
import time
import os
from sklearn.preprocessing import StandardScaler

# Cari file CSV secara otomatis di dalam folder hasil unduhan kagglehub
csv_files = []
for root, dirs, files in os.walk(path):
    for file in files:
        if file.endswith('.csv'):
            csv_files.append(os.path.join(root, file))

if len(csv_files) == 0:
    raise FileNotFoundError("Tidak ditemukan file CSV di dalam folder dataset.")

# Pilih secara spesifik file yang berisi 'songs_with_attributes_and_lyrics'
dataset_file = None
for f in csv_files:
    if 'songs_with_attributes_and_lyrics.csv' in f:
        dataset_file = f
        break

# Jika tidak spesifik ditemukan, cari yang mengandung kata 'attributes'
if dataset_file is None:
    for f in csv_files:
        if 'attributes' in f.lower():
            dataset_file = f
            break

# Jika masih kosong, ambil file yang tidak mengandung 'timestamps'
if dataset_file is None:
    for f in csv_files:
        if 'timestamp' not in f.lower():
            dataset_file = f
            break

# Fallback terakhir jika semua logika di atas gagal
if dataset_file is None:
    dataset_file = csv_files[0]

print(f"Menggunakan file dataset: {dataset_file}")
df = pd.read_csv(dataset_file)

# Memilih fitur numerik
features = ['danceability', 'energy', 'loudness', 'speechiness',
            'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo']

# Menghapus baris yang memiliki nilai kosong jika ada
df_clean = df.dropna(subset=features).reset_index(drop=True)
X = df_clean[features].values

# Standarisasi fitur
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X).astype('float32') # FAISS membutuhkan tipe float32

print(f"Dataset shape setelah dibersihkan: {X_scaled.shape}")

Menggunakan file dataset: /kaggle/input/spotify-songs-with-attributes-and-lyrics/songs_with_attributes_and_lyrics.csv
Dataset shape setelah dibersihkan: (955320, 9)


### Langkah 2: Menentukan Query untuk Pengujian
Kita akan mengambil 100 query acak (atau index tertentu) untuk menguji performa pencarian dan membandingkan hasilnya dengan nilai $k = 10$.

In [23]:
k = 10
# Mengambil 100 sampel data sebagai query untuk perbandingan performa
np.random.seed(42)
query_indices = np.random.choice(len(X_scaled), size=100, replace=False)
queries = X_scaled[query_indices]

### Langkah 3: Exact Nearest Neighbors (Ground Truth)
Menggunakan `NearestNeighbors` dari Scikit-Learn dengan metrik Euclidean sebagai standar kebenaran (exact match).

In [24]:
from sklearn.neighbors import NearestNeighbors

# Inisialisasi dan fitting model
start_build = time.time()
exact_nn = NearestNeighbors(n_neighbors=k, algorithm='brute', metric='euclidean')
exact_nn.fit(X_scaled)
build_time_exact = time.time() - start_build

# Query
start_search = time.time()
exact_distances, exact_indices = exact_nn.kneighbors(queries)
search_time_exact = time.time() - start_search

print(f"Exact NN - Build Time: {build_time_exact:.4f}s, Search Time (100 queries): {search_time_exact:.4f}s")

Exact NN - Build Time: 0.0233s, Search Time (100 queries): 0.4360s


### Langkah 4: ANNOY (Approximate Nearest Neighbors Oh Yeah)
Kita membangun index ANNOY dengan metrik 'euclidean' dan mengukur waktu build serta waktu search.

In [25]:
from annoy import AnnoyIndex

num_features = X_scaled.shape[1]
annoy_index = AnnoyIndex(num_features, 'euclidean')

# Build Index
start_build = time.time()
for i, vector in enumerate(X_scaled):
    annoy_index.add_item(i, vector)
annoy_index.build(n_trees=10) # n_trees memengaruhi akurasi dan kecepatan
build_time_annoy = time.time() - start_build

# Search
start_search = time.time()
annoy_indices = []
for query in queries:
    indices = annoy_index.get_nns_by_vector(query, k)
    annoy_indices.append(indices)
search_time_annoy = time.time() - start_search

annoy_indices = np.array(annoy_indices)
print(f"ANNOY - Build Time: {build_time_annoy:.4f}s, Search Time: {search_time_annoy:.4f}s")

ANNOY - Build Time: 17.1211s, Search Time: 0.0057s


### Langkah 5: FAISS (Facebook AI Similarity Search)
Kita menggunakan index dasar FAISS L2 (`IndexFlatL2`) untuk pencarian cepat berbasis CPU.

In [26]:
import faiss

# Build Index
start_build = time.time()
faiss_index = faiss.IndexFlatL2(num_features)
faiss_index.add(X_scaled)
build_time_faiss = time.time() - start_build

# Search
start_search = time.time()
faiss_distances, faiss_indices = faiss_index.search(queries, k)
search_time_faiss = time.time() - start_search

print(f"FAISS - Build Time: {build_time_faiss:.4f}s, Search Time: {search_time_faiss:.4f}s")

FAISS - Build Time: 0.0587s, Search Time: 0.6082s


### Langkah 6: HNSWLIB
Kita membangun index menggunakan algoritma grafik HNSW dengan metrik L2 (Euclidean).

In [27]:
import hnswlib

# Build Index
start_build = time.time()
hnsw_index = hnswlib.Index(space='l2', dim=num_features)
# Inisialisasi index
hnsw_index.init_index(max_elements=len(X_scaled), ef_construction=200, M=16)
hnsw_index.add_items(X_scaled)
hnsw_index.set_ef(50) # ef berpengaruh pada recall dan kecepatan search
build_time_hnsw = time.time() - start_build

# Search
start_search = time.time()
hnsw_indices, hnsw_distances = hnsw_index.knn_query(queries, k=k)
search_time_hnsw = time.time() - start_search

print(f"HNSWLIB - Build Time: {build_time_hnsw:.4f}s, Search Time: {search_time_hnsw:.4f}s")

HNSWLIB - Build Time: 187.1350s, Search Time: 0.0048s


### Langkah 7: Perbandingan Performa & Akurasi (Recall)
Kita akan membandingkan waktu pembuatan indeks, waktu pencarian, serta menghitung seberapa akurat hasil dari metode Approximate Nearest Neighbor (ANNOY, FAISS, HNSW) dibandingkan dengan Exact NN (Ground Truth).

In [28]:
def calculate_recall(true_indices, pred_indices):
    total_overlap = 0
    for t, p in zip(true_indices, pred_indices):
        total_overlap += len(set(t).intersection(set(p)))
    return total_overlap / (len(true_indices) * k)

recall_annoy = calculate_recall(exact_indices, annoy_indices)
# FAISS mengembalikan exact L2 secara bawaan jika menggunakan IndexFlatL2
recall_faiss = calculate_recall(exact_indices, faiss_indices)
recall_hnsw = calculate_recall(exact_indices, hnsw_indices)

# Menampilkan tabel hasil perbandingan
summary_data = {
    'Metode': ['Exact NN (Sklearn)', 'ANNOY', 'FAISS (FlatL2)', 'HNSWLIB'],
    'Build Time (s)': [build_time_exact, build_time_annoy, build_time_faiss, build_time_hnsw],
    'Search Time (s)': [search_time_exact, search_time_annoy, search_time_faiss, search_time_hnsw],
    'Recall (Akurasi)': [1.0, recall_annoy, recall_faiss, recall_hnsw]
}

df_summary = pd.DataFrame(summary_data)
display(df_summary)

,Metode,Build Time (s),Search Time (s),Recall (Akurasi)
0,Exact NN (Sklearn),0.023318,0.436014,1.000
1,ANNOY,17.121089,0.005725,0.833
2,FAISS (FlatL2),0.058681,0.608193,0.998
3,HNSWLIB,187.134999,0.004806,0.996
